<a href="https://colab.research.google.com/github/AadarshKr-11/pharma-product-matcher/blob/main/pharma_matcher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pharma Product Matcher (Finland, Kela reimbursement data)

Match a messy medicine name (e.g. `burana 400mg tabl 30`) to the exact product in the official Kela list,
using **multilingual embeddings + blocking + business rules**, and measure how well it works.

**Why it matters for pricing:** before comparing prices across sources or countries, you must know it is the *same* product.
A wrong match puts a wrong price into the analysis.

**Pipeline**
```
messy name → normalise → embed → block by active substance → rank by cosine similarity
          → strength rule → pack rule → name rule → threshold → match / not sure / no match
```

**Result (held-out evaluation, system frozen before the run):**
285 unseen products from 22 medicines → **96% precision, 88% recall** at threshold 0.75.

Data: public Kela medicine price data (kela.fi). Personal project; no employer data or tools used.

## 1. Setup

In [16]:
!pip install -q sentence-transformers

In [17]:
import re
import time
from io import StringIO

import numpy as np
import pandas as pd
import requests
from sentence_transformers import SentenceTransformer

## 2. Load and clean the Kela master list

Data-quality issues found and fixed:
- **VNR lost its leading zeros** in Excel → read as text, `zfill(6)`
- **Product type mixed into the name** ("self-care / prescription medicine") → moved to an `Rx` column
- **Invisible zero-width space `\u200b`** inside text (found with `repr()`) → removed
- **Prices ×100** from decimal-comma parsing → validated against Kela and fixed

In [18]:
df = pd.read_excel("finland_KELA.xlsx", dtype={"VNR": str})
df["VNR"] = df["VNR"].str.zfill(6)

df["Rx"] = df["Product"].str.contains("prescription")
df["Product"] = (df["Product"]
                 .str.replace("self-care medicine", "")
                 .str.replace("prescription medicine", "")
                 .str.strip())

for col in ["Product", "Strength", "Dosage Form", "Package Size"]:
    df[col] = df[col].str.replace("\u200b", "")

df = df.reset_index(drop=True)


df["full_name"]  = df["Product"] + " " + df["Strength"] + " " + df["Dosage Form"] + " " + df["Package Size"]
df["embed_name"] = df["Product"] + " " + df["Strength"] + " " + df["Dosage Form"]

print(df.shape)
print(df["Search Product"].value_counts())
print("price range (€):", df["Price"].min(), "-", df["Price"].max())   # sanity check

(344, 16)
Search Product
amlodipin       46
pregabalin      38
burana          30
losartan        29
atorvastatin    29
simvastatin     26
panadol         20
sertralin       18
amorion         14
metformin       13
somac           11
omeprazol       10
seretide         9
pamol            9
zyrtec           7
nexium           7
ventoline        6
nucala           4
primaspan        4
aerius           3
havrix           3
blenrep          2
thyroxin         2
marevan          2
dovato           1
avamys           1
Name: count, dtype: int64
price range (€): 1.55 - 22171.45


## 3. Embed the master list (once)

`paraphrase-multilingual-MiniLM-L12-v2`: 384-dimensional vectors, understands Finnish.
(The English-only `all-MiniLM-L6-v2` was tried first and replaced because the data is Finnish.)

In [19]:
model_ml = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
kela_vectors = model_ml.encode(df["embed_name"].tolist())
print(kela_vectors.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

(344, 384)


## 4. Helper functions

- `cosine`: similarity between two vectors (1 = same meaning)
- `strength`: the mg number in a text (`"400 mg"` → `"400"`)
- `package_size`: the first number in Kela's pack text (`"30 fol ..."` → `"30"`)
- `query_pack`: the pack number in a query = the number that is not the strength (removed once, so `50 mg … 50` works)
- `normalise`: lowercase, `400mg` → `400 mg`, abbreviations → Kela words
- `name_in_query`: every word of a product name appears in the query (hyphen-tolerant, checks raw and normalised query)

In [20]:
def cosine(a, b):
    return (a @ b) / (np.linalg.norm(a) * np.linalg.norm(b))

def strength(text):
    m = re.search(r"(\d+)\s*mg", text)
    if m is None:
        return None
    return m.group(1)

def package_size(text):
    m = re.search(r"(\d+)", text)
    if m is None:
        return None
    return m.group(1)

def query_pack(q):
    nums = re.findall(r"\d+", q)
    mg = strength(q)
    if mg in nums:
        nums.remove(mg)
    if len(nums) == 0:
        return None
    return nums[0]

ABBREV = {"tabl": "tabletti", "tab": "tabletti", "caps": "kapseli", "kaps": "kapseli"}

def normalise(q):
    q = q.lower()
    q = re.sub(r"(\d+)\s*mg", r"\1 mg", q)
    words = q.split()
    words = [ABBREV.get(w, w) for w in words]
    return " ".join(words)

def name_in_query(name, query):
    raw  = query.lower().replace("-", " ").split()
    norm = normalise(query).replace("-", " ").split()
    q_words = set(raw) | set(norm)
    for w in name.lower().replace("-", " ").split():
        if w not in q_words:
            return False
    return True

## 5. Retrieval: `search()` = blocking + ranking

Keeps only products with the same active substance (**blocking**), scores each with cosine similarity, sorts.
Stays broad on purpose, so it can be reused (e.g. for a RAG Q&A layer later).

In [21]:
def search(query, substance, k=10):
    q = model_ml.encode(query)
    block = df[df["Active Substance"] == substance].copy()
    block["score"] = [cosine(q, kela_vectors[i]) for i in block.index]
    return (block.sort_values("score", ascending=False)
                 .head(k)[["Product", "full_name", "Strength", "Package Size", "score"]])

search(normalise("burana 400mg tabl"), "ibuprofeeni", 5)

,Product,full_name,Strength,Package Size,score
28,BURANA,"BURANA 400 mg tabletti, kalvopäällysteinen 100...",400 mg,"100 kpl tablettipurkki, muovi",0.913135
29,BURANA,"BURANA 400 mg tabletti, kalvopäällysteinen 20 ...",400 mg,20 fol läpipainopakkaus,0.913135
26,BURANA,"BURANA 400 mg tabletti, kalvopäällysteinen 10 ...",400 mg,10 fol läpipainopakkaus,0.913135
27,BURANA,"BURANA 400 mg tabletti, kalvopäällysteinen 10 ...",400 mg,"10 kpl tablettipurkki, muovi",0.913135
30,BURANA,"BURANA 400 mg tabletti, kalvopäällysteinen 30 ...",400 mg,30 fol läpipainopakkaus,0.913135


## 6. Matching: `match()` = retrieval + business rules + thresholds

1. normalise the query and retrieve the **whole substance block** (a top-10 cut-off once dropped the right answer: *retrieval recall*)
2. **strength rule**: same mg
3. **pack rule**: same pack count (a 10-pack and a 100-pack have different prices)
4. **name rule**: product name must appear in the query; prefer the most specific name (PANADOL ZAPP over PANADOL)
5. **thresholds**: ≥ 0.75 match · < 0.6 no match · otherwise not sure (goes to review)

In [22]:
MATCH_T, NO_MATCH_T = 0.75, 0.60

def match(query, substance):
    q = normalise(query)
    topCandidates = search(q, substance, 1000)

    q_mg = strength(q)
    if q_mg is not None:
        topCandidates = topCandidates[topCandidates["Strength"].apply(strength) == q_mg]

    q_pack = query_pack(q)
    if q_pack is not None:
        topCandidates = topCandidates[topCandidates["Package Size"].apply(package_size) == q_pack]

    if len(topCandidates) == 0:
        return ("no match", None, 0.0)

    keepPdt = [name_in_query(n, query) for n in topCandidates["Product"]]
    topCandidates = topCandidates[keepPdt].copy()
    if len(topCandidates) == 0:
        return ("no match", None, 0.0)

    topCandidates["name_len"] = topCandidates["Product"].str.split().str.len()
    topCandidates = topCandidates.sort_values(["name_len", "score"], ascending=[False, False])

    top = topCandidates.iloc[0]
    score = float(top["score"])
    if score >= MATCH_T:
        decision = "match"
    elif score < NO_MATCH_T:
        decision = "no match"
    else:
        decision = "not sure"
    return (decision, top["full_name"], round(score, 3))

## 7. Regression tests (re-run after every change)

In [23]:
for q in ["burana 400mg tabl", "burana 200mg tabl", "burana 600mg tabl",
          "burana 300mg tabl", "burana caps 400mg", "burana 400mg tabl 30",
          "burana 400mg tabl 100", "burana 400mg tabl 999"]:
    print(q, "→", match(q, "ibuprofeeni"))

burana 400mg tabl → ('match', 'BURANA 400 mg tabletti, kalvopäällysteinen 100 kpl tablettipurkki, muovi', 0.913)
burana 200mg tabl → ('match', 'BURANA 200 mg tabletti, kalvopäällysteinen 30 fol läpipainopakkaus', 0.913)
burana 600mg tabl → ('match', 'BURANA 600 mg tabletti, kalvopäällysteinen 30 fol läpipainopakkaus', 0.917)
burana 300mg tabl → ('no match', None, 0.0)
burana caps 400mg → ('match', 'BURANA-CAPS 400 mg kapseli, pehmeä 100 fol läpipainopakkaus', 0.815)
burana 400mg tabl 30 → ('match', 'BURANA 400 mg tabletti, kalvopäällysteinen 30 fol läpipainopakkaus', 0.902)
burana 400mg tabl 100 → ('match', 'BURANA 400 mg tabletti, kalvopäällysteinen 100 kpl tablettipurkki, muovi', 0.866)
burana 400mg tabl 999 → ('no match', None, 0.0)


## 8. Test set with known answers

`make_messy()` turns each Kela row into a messy query, so the correct answer is known (synthetic ground truth).
Limitation: generated queries are tidier than real-world mess.

In [24]:
SHORT = {"tabletti": "tabl", "kapseli": "kaps"}

def make_messy(row):
    brand = row["Product"].lower()
    mg = strength(row["Strength"])
    s = f"{mg}mg" if mg is not None else row["Strength"].lower()
    form = row["Dosage Form"].split(",")[0]
    form = SHORT.get(form, form)
    pack = package_size(row["Package Size"])
    return f"{brand} {s} {form} {pack}"

tests = pd.DataFrame({
    "query":     df.apply(make_messy, axis=1),
    "substance": df["Active Substance"],
    "answer":    df["full_name"],
})
tests.head()

,query,substance,answer
0,"avamys 27,5 mikrog/suihke nenäsumute 120",flutikasonifuroaatti,"AVAMYS 27,5 mikrog/suihke nenäsumute, suspensi..."
1,seretide diskus 50/100 mikrog/annos inhalaatio...,salmeteroli ja flutikasoni,SERETIDE DISKUS 50/100 mikrog/annos inhalaatio...
2,seretide diskus 50/100 mikrog/annos inhalaatio...,salmeteroli ja flutikasoni,SERETIDE DISKUS 50/100 mikrog/annos inhalaatio...
3,seretide diskus 50/250 mikrog/annos inhalaatio...,salmeteroli ja flutikasoni,SERETIDE DISKUS 50/250 mikrog/annos inhalaatio...
4,seretide diskus 50/250 mikrog/annos inhalaatio...,salmeteroli ja flutikasoni,SERETIDE DISKUS 50/250 mikrog/annos inhalaatio...


## 9. Run the matcher on every test query and mark it

"Correct" = same product, strength, form and **pack count** (blister vs bottle of the same count had identical prices in the data).

In [25]:
decisions, predicted, scores = [], [], []
for q, s in zip(tests["query"], tests["substance"]):
    d, p, sc = match(q, s)
    decisions.append(d); predicted.append(p); scores.append(sc)
tests["decision"], tests["predicted"], tests["score"] = decisions, predicted, scores

df["key"] = df["embed_name"] + " | " + df["Package Size"].apply(package_size)
key_of = dict(zip(df["full_name"], df["key"]))

correct_list = []
for p, a in zip(tests["predicted"], tests["answer"]):
    if p is None:
        correct_list.append(False)
    else:
        correct_list.append(key_of[p] == key_of[a])
tests["correct"] = correct_list

## 10. Results: dev set vs unseen test set

The system was **frozen** (rules + threshold 0.75) before the unseen products were scraped, and evaluated on them once.

In [26]:
DEV = ["burana", "panadol", "pamol"]
is_new = ~df["Search Product"].str.lower().isin(DEV)
unseen, dev = tests[is_new], tests[~is_new]

def report(t, name):
    m = t[t["decision"] == "match"]
    print(f"--- {name}: {len(t)} queries ---")
    print(t["decision"].value_counts().to_string())
    print(f"Precision: {m['correct'].mean():.0%}")
    print(f"Recall:    {m['correct'].sum() / len(t):.0%}")
    print()

report(dev, "DEV (seen)")
report(unseen, "UNSEEN (new products)")

--- DEV (seen): 59 queries ---
decision
match       57
not sure     2
Precision: 100%
Recall:    97%

--- UNSEEN (new products): 285 queries ---
decision
match       262
no match     16
not sure      7
Precision: 96%
Recall:    88%



## 11. Error analysis (unseen)

The 11 wrong matches on unseen products fall into two patterns:
- **10 combination strengths** (`5/160 mg` vs `10/160 mg`; `100/12,5 mg` vs `50/12,5 mg`) → the strength regex reads only one number (and splits decimals at the comma), so the rule can't separate variants
- **1 device variant** (Nucala pre-filled pen vs pre-filled syringe, same strength and pack)

None of the errors came from retrieval, blocking, the pack rule or the name rule.

In [27]:
bad = unseen[(unseen["decision"] == "match") & (unseen["correct"] == False)]
print(len(bad), "wrong matches on unseen")
bad[["query", "answer", "predicted", "score"]]

11 wrong matches on unseen


,query,answer,predicted,score
11,nucala biologic 100mg injektioneste 1,"NUCALA Biologic 100 mg 1 ml, kiinteä neula, tu...","NUCALA Biologic 100 mg 1 ml, kiinteä neula inj...",0.796
161,amlodipin/valsartan krka 160mg tabl 100,"AMLODIPIN/VALSARTAN KRKA 5/160 mg tabletti, ka...","AMLODIPIN/VALSARTAN KRKA 10/160 mg tabletti, k...",0.834
162,amlodipin/valsartan krka 160mg tabl 30,"AMLODIPIN/VALSARTAN KRKA 5/160 mg tabletti, ka...","AMLODIPIN/VALSARTAN KRKA 10/160 mg tabletti, k...",0.840
167,amlodipin/valsartan orion 160mg tabl 98,"AMLODIPIN/VALSARTAN ORION 5/160 mg tabletti, k...","AMLODIPIN/VALSARTAN ORION 10/160 mg tabletti, ...",0.921
171,amlodipin/valsartan stada 160mg tabl 100,"AMLODIPIN/VALSARTAN STADA 5/160 mg tabletti, k...","AMLODIPIN/VALSARTAN STADA 10/160 mg tabletti, ...",0.902
174,amlodipin/valsartan/hydrochlorothiazide orion ...,AMLODIPIN/VALSARTAN/HYDROCHLOROTHIAZIDE ORION ...,AMLODIPIN/VALSARTAN/HYDROCHLOROTHIAZIDE ORION ...,0.879
182,amlodipine/valsartan/hydrochlorothiazide krka ...,AMLODIPINE/VALSARTAN/HYDROCHLOROTHIAZIDE KRKA ...,AMLODIPINE/VALSARTAN/HYDROCHLOROTHIAZIDE KRKA ...,0.842
185,amlodipine/valsartan/hydrochlorothiazide krka ...,AMLODIPINE/VALSARTAN/HYDROCHLOROTHIAZIDE KRKA ...,AMLODIPINE/VALSARTAN/HYDROCHLOROTHIAZIDE KRKA ...,0.827
187,amlodipine/valsartan/hydrochlorothiazide stada...,AMLODIPINE/VALSARTAN/HYDROCHLOROTHIAZIDE STADA...,AMLODIPINE/VALSARTAN/HYDROCHLOROTHIAZIDE STADA...,0.889
207,losartan/hydrochlorothiazide krka 5mg tabl 28,"LOSARTAN/HYDROCHLOROTHIAZIDE KRKA 100/12,5 mg ...","LOSARTAN/HYDROCHLOROTHIAZIDE KRKA 50/12,5 mg t...",0.839


## 12. Version history (dev set)

| Version | Change | Precision | Recall |
|---|---|---|---|
| v1 | first run | 88% | 65% |
| v2 | pack bug fix + full product names in tests | 92% | 67% |
| v3 | name rule (brand / variant) | 97% | 67% |
| v4 | retrieve the whole substance block (retrieval recall) | 100% | 67% |
| v4 @ 0.75 | threshold chosen before the unseen run | 100% | 97% |

**Unseen test (285 products, 22 medicines): 95% precision, 88% recall.**

In [28]:
# Extra check: does packaging type (blister vs bottle) change price for the same pack count?
df["pack"] = df["Package Size"].apply(package_size)
check = df.groupby(["Product", "Strength", "pack"])["Price"].agg(["count", "nunique", "min", "max"])
check[check["count"] > 1]

count  nunique    min    max
Product                 Strength pack                              
ATORVASTATIN RATIOPHARM 10 mg    100       2        2   2.27   2.77
                        20 mg    100       2        2   2.54   3.03
                        40 mg    100       2        2   4.09   4.57
BURANA                  400 mg   10        2        1   3.97   3.97
                                 30        2        1   9.99   9.99
PRIMASPAN               100 mg   100       2        1  11.76  11.76
SERTRALIN ORION         100 mg   100       2        1  19.76  19.76
                        50 mg    100       2        1  14.46  14.46
SOMAC                   40 mg    100       2        1   6.32   6.32
ZYRTEC                  10 mg    30        2        2  13.68  16.80
                        10 mg/ml 20        3        3  18.55  21.84